# Tugas 1 – Data Preprocessing
**Nama:** Muhammad Endi Saputra Ramadhan  
**NIM:** 3022410040  
**Partner:** Muhammad Adi Indrawan – 3022410001  
**Kasus:** E-commerce Customer Behavior (MegaStore) – 1.600 baris × 17 variabel  

Setiap tahap dijelaskan dengan format **Input → Proses → Output**.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.decomposition import PCA
pd.set_option('display.max_columns', None)

## 2. Membaca Dataset
**Input:** CSV dari partner (GitHub). **Proses:** `pd.read_csv`. **Output:** DataFrame awal.

Ganti `USERNAME` dengan username GitHub Anda. Jika gagal, notebook meminta upload file manual.

In [ ]:
url = 'https://raw.githubusercontent.com/USERNAME/penambangan-data/main/tugas-1/ecommerce_customer_behavior_1.csv'
try:
    data = pd.read_csv(url)
except Exception:
    from google.colab import files
    up = files.upload()
    data = pd.read_csv(list(up.keys())[0])

print("DATA AWAL")
display(data.head(8))
print(data.shape)
data.info()

## 3. Eksplorasi Data (EDA)
**Proses:** cek missing value, duplikat, nilai unik kategorik, statistik deskriptif, dan skewness.

In [ ]:
miss_awal = pd.DataFrame({'Jumlah NaN': data.isnull().sum(), 'Persen (%)': (data.isnull().mean()*100).round(2)})
display(miss_awal[miss_awal['Jumlah NaN'] > 0])
print('Baris duplikat persis :', data.duplicated().sum())

kat_cols = data.select_dtypes(exclude='number').columns.drop(['customer_id','session_id'])
for c in kat_cols:
    print(f"{c:18s}", data[c].dropna().unique().tolist())

num_cols = ['age','session_duration_minutes','pages_viewed','product_price']
display(data[num_cols].describe().round(2))
print('Skewness:'); display(data[num_cols].skew().round(2))

## 4. Duplikat dan Atribut Tidak Relevan
Baris duplikat persis = 0, tetapi ada `session_id` yang kembar. Kita periksa apakah itu duplikat sebenarnya.

In [ ]:
print('customer_id unik:', data['customer_id'].is_unique)
dup_sess = data[data['session_id'].duplicated(keep=False)].sort_values('session_id')
display(dup_sess[['customer_id','session_id','age','gender','location','product_category','product_price']])

**Keputusan:** pasangan `session_id` kembar berasal dari pelanggan berbeda dengan isi berbeda (bukan duplikat, hanya tabrakan ID), sehingga **tidak dihapus**. `customer_id` dan `session_id` hanya identitas, jadi dibuang sebagai atribut tidak relevan.

In [ ]:
df = data.drop_duplicates().drop(columns=['customer_id','session_id'])
print('Setelah drop duplikat & ID:', df.shape)

## 5. Konsistensi Teks dan Logika Antar-Kolom

In [ ]:
for c in kat_cols:
    s = df[c].dropna()
    print(f"{c:18s} spasi berlebih: {(s != s.str.strip()).sum()} | variasi kapital: {s.str.lower().nunique() != s.nunique()}")

# Logika: pembelian hanya sah jika barang dimasukkan keranjang dan keranjang tidak ditinggalkan
pelanggaran = ((df['purchase_made']=='Yes') & ((df['add_to_cart']=='No') | (df['abandoned_cart']=='Yes'))).sum()
print('Pelanggaran logika purchase/cart:', pelanggaran)
print('Nilai tidak valid (<=0) numerik :', (df[num_cols] <= 0).sum().sum())
print('Usia di luar 0-100              :', ((df['age']<0)|(df['age']>100)).sum())

## 6. Missing Value
### 6.1 payment_method: missing struktural
**Proses:** bandingkan NaN pada `payment_method` dengan `purchase_made`.

In [ ]:
display(pd.crosstab(df['purchase_made'], df['payment_method'].isna(), rownames=['purchase_made'], colnames=['payment_method kosong']))
df['payment_method'] = df['payment_method'].fillna('No Purchase')
print(df['payment_method'].value_counts())

`payment_method` kosong **tepat** ketika `purchase_made = No` (tidak ada pembayaran), sehingga bukan data hilang sebenarnya. Nilainya diisi kategori `No Purchase`, bukan modus.

### 6.2 Missing value sebenarnya: statistik deskriptif
**Input:** age, session_duration_minutes, pages_viewed, product_price (numerik) dan traffic_source (kategorik).

In [ ]:
stat_num = pd.DataFrame({'mean': df[num_cols].mean(), 'median': df[num_cols].median(), 'std': df[num_cols].std(),
                         'skewness': df[num_cols].skew(), 'jumlah NaN': df[num_cols].isnull().sum()})
display(stat_num.round(3))
print('Modus traffic_source :', df['traffic_source'].mode()[0])
print(df['traffic_source'].value_counts(dropna=False))
sebelum = df[num_cols].describe().T[['count','mean','50%','std']]

**Keputusan:** `age` simetris (skewness ≈ 0, mean ≈ median) → diisi **mean**. `session_duration_minutes`, `pages_viewed`, `product_price` miring ke kanan (skewness > 1,5) → diisi **median** karena mean tertarik nilai ekstrem. `traffic_source` (kategorik) → diisi **modus**.

In [ ]:
df['age'] = df['age'].fillna(df['age'].mean())
for c in ['session_duration_minutes','pages_viewed','product_price']:
    df[c] = df[c].fillna(df[c].median())
df['traffic_source'] = df['traffic_source'].fillna(df['traffic_source'].mode()[0])

print('Sisa missing value:', df.isnull().sum().sum())
sesudah = df[num_cols].describe().T[['count','mean','50%','std']]
display(pd.concat([sebelum.add_suffix(' (sblm)'), sesudah.add_suffix(' (ssdh)')], axis=1).round(3))

## 7. Outlier (IQR + Capping)
**Proses:** batas = Q1 − 1,5×IQR dan Q3 + 1,5×IQR; nilai di luar batas dipotong ke batas terdekat.

In [ ]:
def batas(s):
    q1, q3 = s.quantile(.25), s.quantile(.75); i = q3 - q1
    return q1 - 1.5*i, q3 + 1.5*i

out_tbl = pd.DataFrame([[c, *batas(df[c]), int(((df[c] < batas(df[c])[0]) | (df[c] > batas(df[c])[1])).sum())] for c in num_cols],
                       columns=['variabel','batas_bawah','batas_atas','jumlah_outlier']).round(2)
display(out_tbl)

df[num_cols].plot(kind='box', subplots=True, layout=(1,4), figsize=(12,4), title='Boxplot SEBELUM capping')
plt.tight_layout(); plt.show()

skew_sebelum = df[num_cols].skew()
df_clean = df.copy()
for c in num_cols:
    lo, hi = batas(df_clean[c]); df_clean[c] = df_clean[c].clip(lo, hi)

df_clean[num_cols].plot(kind='box', subplots=True, layout=(1,4), figsize=(12,4), title='Boxplot SESUDAH capping')
plt.tight_layout(); plt.show()
display(pd.DataFrame({'skew sebelum': skew_sebelum, 'skew sesudah': df_clean[num_cols].skew()}).round(3))

Nilai-nilai ini bukan kesalahan input (semua > 0 dan masuk akal), melainkan sesi/harga yang memang panjang atau mahal, sehingga dipotong (capping), tidak dihapus.

## 8. Encoding Data Kategorik

In [ ]:
enc = df_clean.copy()
biner = ['discount_applied','add_to_cart','abandoned_cart','purchase_made']
for c in biner: enc[c] = enc[c].map({'No':0, 'Yes':1})
nominal = ['gender','location','device_type','traffic_source','product_category','payment_method','time_of_day']
enc = pd.get_dummies(enc, columns=nominal, dtype=int)
print(enc.shape); display(enc.head())

# DATA TRANSFORMATION
Variabel: **age** (sesuai contoh materi) dan **product_price** (skala besar).

In [ ]:
t = df_clean[['age','product_price']].copy()
age = t['age'].copy()
print('DATA AWAL'); display(t.head())

### Decimal Scaling
v' = v / 10^j, j = jumlah digit nilai maksimum.

In [ ]:
for c in ['age','product_price']:
    j = len(str(int(t[c].abs().max())))
    t[c+'_Decimal'] = t[c] / (10**j)
    print(c, '-> j =', j)
display(t.head())

### Min-Max Scaling (Manual & Library)

In [ ]:
t['Age_MinMax_Manual']  = (age - age.min()) / (age.max() - age.min())
t['Age_MinMax_Library'] = MinMaxScaler().fit_transform(age.to_frame()).ravel()
display(t[['age','Age_MinMax_Manual','Age_MinMax_Library']].head())

### Standardization (Manual & Library)
Manual memakai `std()` pandas (n−1), library memakai n, sehingga ada selisih sangat kecil.

In [ ]:
t['Age_Standard_Manual']  = ((age - age.mean()) / age.std()).round(3)
t['Age_Standard_Library'] = StandardScaler().fit_transform(age.to_frame()).ravel()
display(t[['age','Age_Standard_Manual','Age_Standard_Library']].head())

### Robust Scaling (Manual & Library)

In [ ]:
q1, q3 = age.quantile(0.25), age.quantile(0.75); iqr = q3 - q1
t['Age_Robust_Manual']  = (age - age.median()) / iqr
t['Age_Robust_Library'] = RobustScaler().fit_transform(age.to_frame()).ravel()
display(t[['age','Age_Robust_Manual','Age_Robust_Library']].head())

### Perbandingan pada product_price (miring ke kanan)

In [ ]:
p = df_clean['product_price']
t['Price_MinMax']   = MinMaxScaler().fit_transform(p.to_frame()).ravel()
t['Price_Standard'] = StandardScaler().fit_transform(p.to_frame()).ravel()
t['Price_Robust']   = RobustScaler().fit_transform(p.to_frame()).ravel()
print("HASIL AKHIR DATA TRANSFORMATION")
display(t.describe().round(3).T[['mean','std','min','max']])

# PCA (Principal Component Analysis)
**Input:** 4 fitur numerik bersih. **Proses:** standardisasi → PCA. **Output:** explained variance.

In [ ]:
x = df_clean[num_cols]
X = pd.DataFrame(StandardScaler().fit_transform(x), columns=x.columns)
display(X.head())

model = PCA(); results = model.fit(X)
ratio = results.explained_variance_ratio_; cum = np.cumsum(ratio)
pca_tbl = pd.DataFrame({'Eigenvalue': results.explained_variance_, 'Varians (%)': ratio*100, 'Kumulatif (%)': cum*100},
                       index=[f'PC{i+1}' for i in range(len(ratio))])
display(pca_tbl.round(3))

plt.plot(range(1, len(ratio)+1), ratio*100, '-ob', label='Per komponen')
plt.plot(range(1, len(ratio)+1), cum*100, '-or', label='Kumulatif')
plt.axhline(80, ls='--', color='gray')
plt.xlabel('number of components'); plt.ylabel('percentage variance explained'); plt.legend(); plt.show()

k = int(np.argmax(cum >= 0.80) + 1)
print('Jumlah komponen untuk >= 80% varians:', k)
loading = pd.DataFrame(results.components_.T, index=num_cols, columns=[f'PC{i+1}' for i in range(len(ratio))])
display(loading.round(3))

## 9. Menyimpan Dataset Bersih

In [ ]:
enc.to_csv('ecommerce_customer_behavior_bersih.csv', index=False)
print('Tersimpan:', enc.shape)

## Kesimpulan
Masalah nyata pada dataset: missing value (numerik, kategorik, dan struktural pada `payment_method`), tabrakan `session_id`, distribusi miring dengan outlier, atribut identitas tidak relevan, dan perbedaan skala. Semuanya ditangani, dan data akhir siap untuk pemodelan.